In [1]:
# 3.11.9

import ee
import geemap
import os

ee.Initialize()

In [2]:
import pandas as pd

uscities = pd.read_excel("C:/Users/adamk/Downloads/simplemaps_uscities_basicv1.93/uscities.xlsx")

uscities = uscities.drop_duplicates(subset='city')

cities = {
    row.city: (row.lng, row.lat)
    for _, row in uscities.iterrows()
}

print(list(cities.items())[:5])

[('New York', (-73.9249, 40.6943)), ('Los Angeles', (-118.4068, 34.1141)), ('Chicago', (-87.6866, 41.8375)), ('Miami', (-80.2101, 25.784)), ('Houston', (-95.3885, 29.786))]


In [3]:
import ee
import geemap

def get_image(lon, lat, year):
    point = ee.Geometry.Point(lon, lat)
    
    collection = (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterBounds(point)
        .filterDate(f"{year}-01-01", f"{year}-12-31")
        .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 70))
        .select(["B4", "B3", "B2"])
    )

    if collection.size().getInfo() == 0:
        return None
    
    image = collection.median()

    rgb = image.visualize(
        bands=["B4", "B3", "B2"],
        min=0,
        max=3000
    )

    return rgb.clip(point.buffer(1500))

In [4]:
import ee
import geemap

def safe_get(lon, lat, year):
    img = get_image(lon, lat, year)

    if img is not None:
        return img

    print(f"Fallback: relaxing cloud filter for {year}")

    point = ee.Geometry.Point(lon, lat)

    collection = (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterBounds(point)
        .filterDate(f"{year}-01-01", f"{year}-12-31")
        .select(["B4", "B3", "B2"])
    )

    if collection.size().getInfo() == 0:
        return None

    image = collection.median().visualize(
        bands=["B4", "B3", "B2"],
        min=0,
        max=3000
    )

    return image.clip(point.buffer(1500))

In [5]:
import ee
import geemap

def export_image(image, city, year, lon, lat):
    if image is None:
        print(f"Skipping {city} {year}")
        return

    task = ee.batch.Export.image.toDrive(
        image=image,
        description=f"{city}_{year}",
        folder="GEE_Sat_Exports",
        fileNamePrefix=f"{city}_{year}",
        scale=30,
        region=ee.Geometry.Point(lon, lat).buffer(1500).bounds(),
        maxPixels=1e13
    )

    task.start()
    print(f"Started export: {city} {year}")

In [20]:
# 3.11.9

cities_small = dict(list(cities.items())[:100])

for city, (lon, lat) in cities_small.items():
    img_2015 = get_image(lon, lat, 2015)
    img_2020 = get_image(lon, lat, 2020)

    export_image(img_2015, city, 2015, lon, lat)
    export_image(img_2020, city, 2020, lon, lat)

Started export: New York 2015
Started export: New York 2020
Started export: Los Angeles 2015
Started export: Los Angeles 2020
Started export: Chicago 2015
Started export: Chicago 2020
Started export: Miami 2015
Started export: Miami 2020
Started export: Houston 2015
Started export: Houston 2020
Started export: Dallas 2015
Started export: Dallas 2020
Started export: Philadelphia 2015
Started export: Philadelphia 2020
Started export: Atlanta 2015
Started export: Atlanta 2020
Started export: Washington 2015
Started export: Washington 2020
Started export: Boston 2015
Started export: Boston 2020
Started export: Phoenix 2015
Started export: Phoenix 2020
Started export: Detroit 2015
Started export: Detroit 2020
Started export: Seattle 2015
Started export: Seattle 2020
Started export: San Francisco 2015
Started export: San Francisco 2020
Started export: San Diego 2015
Started export: San Diego 2020
Started export: Tampa 2015
Started export: Tampa 2020
Started export: Minneapolis 2015
Started e

In [19]:
import ee
ee.Initialize()

for t in ee.batch.Task.list():
    print(t.config.get("description"), "→", t.status())

Boise_2025 → {'state': 'READY', 'description': 'Boise_2025', 'priority': 100, 'creation_timestamp_ms': 1776975424037, 'update_timestamp_ms': 1776975424037, 'start_timestamp_ms': 0, 'task_type': 'EXPORT_IMAGE', 'id': 'I2Z6ZDSLRCOHTRUP2MQYT4JK', 'name': 'projects/286197774775/operations/I2Z6ZDSLRCOHTRUP2MQYT4JK'}
Boise_2015 → {'state': 'READY', 'description': 'Boise_2015', 'priority': 100, 'creation_timestamp_ms': 1776975423748, 'update_timestamp_ms': 1776975423748, 'start_timestamp_ms': 0, 'task_type': 'EXPORT_IMAGE', 'id': 'CXGBG7DAGDJFSBG7XTVYUWYI', 'name': 'projects/286197774775/operations/CXGBG7DAGDJFSBG7XTVYUWYI'}
Bonita Springs_2025 → {'state': 'READY', 'description': 'Bonita Springs_2025', 'priority': 100, 'creation_timestamp_ms': 1776975422455, 'update_timestamp_ms': 1776975422455, 'start_timestamp_ms': 0, 'task_type': 'EXPORT_IMAGE', 'id': '72Y43YRYGFWNA6MS6HLTYVCO', 'name': 'projects/286197774775/operations/72Y43YRYGFWNA6MS6HLTYVCO'}
Bonita Springs_2015 → {'state': 'READY', 'd

In [1]:
#from PIL import Image
#import matplotlib.pyplot as plt

#img = Image.open("C:/Users/adamk/Downloads/Atlanta_2015.tif")

#plt.imshow(img)
#plt.axis("off")
##plt.show()

In [ ]:
# 3.11.9

import os

os.makedirs(f"C:/Users/adamk/Downloads/SatelliteImages", exist_ok=True)

for city in cities_small:
    city_lower = city.lower()
    os.makedirs(f"C:/Users/adamk/Downloads/SatelliteImages/{city_lower}/2015", exist_ok=True)
    os.makedirs(f"C:/Users/adamk/Downloads/SatelliteImages/{city_lower}/2020", exist_ok=True)

In [10]:
import os
import shutil

# Path where your unsorted images are
source_folder = "C:/Users/adamk/Downloads/GEE_Sat_Exports-20260423T161446Z-3-001/GEE_Sat_Exports"

# Destination root folder
destination_root = "C:/Users/adamk/Downloads/SatelliteImages"

# Supported image extensions
valid_extensions = (".png", ".jpg", ".jpeg", ".tif", ".tiff")

for filename in os.listdir(source_folder):
    # Skip non-image files
    if not filename.lower().endswith(valid_extensions):
        continue

    try:
        # Remove extension
        name_without_ext = os.path.splitext(filename)[0]

        # Split into city and year
        city, year = name_without_ext.rsplit("_", 1)

        # Clean up (just in case)
        city = city.strip()
        year = year.strip()

        # Create destination path
        city_folder = os.path.join(destination_root, city)
        year_folder = os.path.join(city_folder, year)

        os.makedirs(year_folder, exist_ok=True)

        # Move file
        src_path = os.path.join(source_folder, filename)
        dst_path = os.path.join(year_folder, filename)

        shutil.move(src_path, dst_path)

        print(f"Moved: {filename} → {city}/{year}/")

    except ValueError:
        print(f"Skipping (invalid format): {filename}")

print("Done organizing images.")

Moved: Atlanta_2015.tif → Atlanta/2015/
Moved: Atlanta_2020.tif → Atlanta/2020/
Moved: Boston_2015.tif → Boston/2015/
Moved: Boston_2020.tif → Boston/2020/
Moved: Chicago_2015.tif → Chicago/2015/
Moved: Chicago_2020.tif → Chicago/2020/
Moved: Dallas_2015.tif → Dallas/2015/
Moved: Dallas_2020.tif → Dallas/2020/
Moved: Houston_2015.tif → Houston/2015/
Moved: Houston_2020.tif → Houston/2020/
Moved: Los Angeles_2015.tif → Los Angeles/2015/
Moved: Los Angeles_2020.tif → Los Angeles/2020/
Moved: Miami_2015.tif → Miami/2015/
Moved: Miami_2020.tif → Miami/2020/
Moved: New York_2015.tif → New York/2015/
Moved: New York_2020.tif → New York/2020/
Moved: Philadelphia_2015.tif → Philadelphia/2015/
Moved: Philadelphia_2020.tif → Philadelphia/2020/
Moved: Washington_2015.tif → Washington/2015/
Moved: Washington_2020.tif → Washington/2020/
Done organizing images.


In [ ]:
# 3.11.14

import os
from PIL import Image
import torch
from torch.utils.data import Dataset
from torchvision import transforms

class SatelliteDataset(Dataset):
    def __init__(self, root_dir):
        self.pairs = []
        self.transform = transforms.Compose([
            transforms.Resize((128, 128)),
            transforms.ToTensor()
        ])

        for city in os.listdir(root_dir):
            city_path = os.path.join(root_dir, city)
            past_dir = os.path.join(city_path, "2015")
            future_dir = os.path.join(city_path, "2020")

            for img in os.listdir(past_dir):
                self.pairs.append((
                    os.path.join(past_dir, img),
                    os.path.join(future_dir, img)
                ))

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        past, future = self.pairs[idx]
        past_img = self.transform(Image.open(past).convert("RGB"))
        future_img = self.transform(Image.open(future).convert("RGB"))
        return past_img, future_img

In [51]:
import torch.nn as nn

class Generator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 64, 4, 2, 1),
            nn.ReLU(),

            nn.Conv2d(64, 128, 4, 2, 1),
            nn.BatchNorm2d(128),
            nn.ReLU(),

            nn.ConvTranspose2d(128, 64, 4, 2, 1),
            nn.ReLU(),

            nn.ConvTranspose2d(64, 3, 4, 2, 1),
            nn.Tanh()
        )

    def forward(self, x):
        return self.net(x)

In [52]:
import torch.nn as nn

class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(6, 64, 4, 2, 1),
            nn.LeakyReLU(0.2),

            nn.Conv2d(64, 128, 4, 2, 1),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2),

            nn.Flatten(),
            nn.Linear(128 * 32 * 32, 1)
        )

    def forward(self, past, future):
        x = torch.cat([past, future], dim=1)
        return self.net(x)

In [ ]:
from PIL import Image
from torchvision import transforms

# Create dataset
dataset = SatelliteDataset(root_dir="C:/Users/adamk/Downloads/SatelliteImages")

print("Dataset length:", len(dataset))


# ✅ THIS MUST BE INSIDE THE CLASS (not here)
# So we are assuming you will paste this into SatelliteDataset

class SatelliteDataset(Dataset):
    def __init__(self, root_dir):
        self.pairs = []
        self.transform = transforms.Compose([
            transforms.Resize((128, 128)),
            transforms.ToTensor()
        ])

        valid_ext = (".tif", ".png", ".jpg", ".jpeg")

        for city in os.listdir(root_dir):
            city_path = os.path.join(root_dir, city)
            past_dir = os.path.join(city_path, "2015")
            future_dir = os.path.join(city_path, "2020")

            if not os.path.exists(past_dir) or not os.path.exists(future_dir):
                continue

            past_images = sorted([
                f for f in os.listdir(past_dir)
                if f.lower().endswith(valid_ext)
            ])

            future_images = sorted([
                f for f in os.listdir(future_dir)
                if f.lower().endswith(valid_ext)
            ])

            for p, f in zip(past_images, future_images):
                self.pairs.append((
                    os.path.join(past_dir, p),
                    os.path.join(future_dir, f)
                ))

    def __len__(self):
        return len(self.pairs)

    # ✅ FIXED: this belongs inside the class
    def __getitem__(self, idx):
        past, future = self.pairs[idx]

        print("Loading past:", past)
        print("Loading future:", future)

        past_img = Image.open(past).convert("RGB")
        future_img = Image.open(future).convert("RGB")

        return self.transform(past_img), self.transform(future_img)


# ✅ Now this works correctly
dataset = SatelliteDataset(root_dir="C:/Users/adamk/Downloads/SatelliteImages")

print("Dataset length:", len(dataset))

# Debug single sample
past, future = dataset[0]
print(past.shape, future.shape)

Dataset length: 10
Dataset length: 10
Loading past: C:/Users/adamk/Downloads/SatelliteImages\Atlanta\2015\Atlanta_2015.tif
Loading future: C:/Users/adamk/Downloads/SatelliteImages\Atlanta\2020\Atlanta_2020.tif
torch.Size([3, 128, 128]) torch.Size([3, 128, 128])


In [55]:
import torch
import torch.optim as optim
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

G = Generator().to(device)
D = Discriminator().to(device)

criterion = nn.BCEWithLogitsLoss()
g_optimizer = optim.Adam(G.parameters(), lr=2e-4)
d_optimizer = optim.Adam(D.parameters(), lr=2e-4)

from torch.utils.data import DataLoader

dataset = SatelliteDataset(root_dir="C:/Users/adamk/Downloads/SatelliteImages")

dataloader = DataLoader(
    dataset,
    batch_size=16,
    shuffle=True,
    num_workers=0
)

for epoch in range(500):
    total_loss = 0
    for past, real_future in dataloader:
        past = past.to(device)
        real_future = real_future.to(device)

        # Train Discriminator
        fake_future = G(past)

        loss = criterion(fake_future, real_future)

        total_loss += loss.item()
        

        real_labels = torch.ones(past.size(0), 1).to(device)
        fake_labels = torch.zeros(past.size(0), 1).to(device)

        d_real = D(past, real_future)
        d_fake = D(past, fake_future)

        d_loss = criterion(d_real, real_labels) + criterion(d_fake, fake_labels)

        d_optimizer.zero_grad()
        d_loss.backward()
        d_optimizer.step()

        # Train Generator
        fake_future = G(past)
        d_fake = D(past, fake_future)
        g_loss = criterion(d_fake, real_labels)

        g_optimizer.zero_grad()
        g_loss.backward()
        g_optimizer.step()

    print(f"Epoch {epoch} | D Loss: {d_loss.item()} | G Loss: {g_loss.item()}")

Loading past: C:/Users/adamk/Downloads/SatelliteImages\Miami\2015\Miami_2015.tif
Loading future: C:/Users/adamk/Downloads/SatelliteImages\Miami\2020\Miami_2020.tif
Loading past: C:/Users/adamk/Downloads/SatelliteImages\Dallas\2015\Dallas_2015.tif
Loading future: C:/Users/adamk/Downloads/SatelliteImages\Dallas\2020\Dallas_2020.tif
Loading past: C:/Users/adamk/Downloads/SatelliteImages\Boston\2015\Boston_2015.tif
Loading future: C:/Users/adamk/Downloads/SatelliteImages\Boston\2020\Boston_2020.tif
Loading past: C:/Users/adamk/Downloads/SatelliteImages\Philadelphia\2015\Philadelphia_2015.tif
Loading future: C:/Users/adamk/Downloads/SatelliteImages\Philadelphia\2020\Philadelphia_2020.tif
Loading past: C:/Users/adamk/Downloads/SatelliteImages\Chicago\2015\Chicago_2015.tif
Loading future: C:/Users/adamk/Downloads/SatelliteImages\Chicago\2020\Chicago_2020.tif
Loading past: C:/Users/adamk/Downloads/SatelliteImages\Washington\2015\Washington_2015.tif
Loading future: C:/Users/adamk/Downloads/Sate

In [56]:
# Visual Comparison

import matplotlib.pyplot as plt

def show_images(past, fake, real):
    fig, axs = plt.subplots(1, 3)
    axs[0].imshow(past.permute(1,2,0).cpu())
    axs[0].set_title("Past")

    axs[1].imshow(fake.permute(1,2,0).cpu())
    axs[1].set_title("Generated")

    axs[2].imshow(real.permute(1,2,0).cpu())
    axs[2].set_title("Real Future")

    plt.show()

In [57]:
%pip install scikit-image

from skimage.metrics import structural_similarity as ssim
import numpy as np
import torch.nn as nn

criterion = nn.L1Loss()   # better than MSE for image generation

import torch.optim as optim

optimizer = optim.Adam(G.parameters(), lr=2e-4, betas=(0.5, 0.999))

G.train()


for past, real_future in dataloader:
    past = past.to(device)
    real_future = real_future.to(device)

    fake_future = G(past)

    loss = criterion(fake_future, real_future)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()



from torch.utils.data import DataLoader, TensorDataset

def create_dataloader(past_images, future_images, batch_size=8):
    dataset = TensorDataset(
        torch.stack(past_images),
        torch.stack(future_images)
    )

    return DataLoader(dataset, batch_size=batch_size, shuffle=True)

Note: you may need to restart the kernel to use updated packages.
Loading past: C:/Users/adamk/Downloads/SatelliteImages\Chicago\2015\Chicago_2015.tif
Loading future: C:/Users/adamk/Downloads/SatelliteImages\Chicago\2020\Chicago_2020.tif
Loading past: C:/Users/adamk/Downloads/SatelliteImages\Atlanta\2015\Atlanta_2015.tif
Loading future: C:/Users/adamk/Downloads/SatelliteImages\Atlanta\2020\Atlanta_2020.tif
Loading past: C:/Users/adamk/Downloads/SatelliteImages\Houston\2015\Houston_2015.tif
Loading future: C:/Users/adamk/Downloads/SatelliteImages\Houston\2020\Houston_2020.tif
Loading past: C:/Users/adamk/Downloads/SatelliteImages\Miami\2015\Miami_2015.tif
Loading future: C:/Users/adamk/Downloads/SatelliteImages\Miami\2020\Miami_2020.tif
Loading past: C:/Users/adamk/Downloads/SatelliteImages\Los Angeles\2015\Los Angeles_2015.tif
Loading future: C:/Users/adamk/Downloads/SatelliteImages\Los Angeles\2020\Los Angeles_2020.tif
Loading past: C:/Users/adamk/Downloads/SatelliteImages\Philadelphi

In [ ]:
import os

root = "C:/Users/adamk/Downloads/SatelliteImages/Chicago"

print("2015 files:")
print(os.listdir(os.path.join(root, "2015")))

print("\n2020 files:")
print(os.listdir(os.path.join(root, "2020")))

2015 files:
['Chicago_2015.tif']

2020 files:
['Chicago_2020.tif']


In [ ]:
root_dir = "C:/Users/adamk/Downloads/SatelliteImages/"

import os
from PIL import Image

def load_city_images(root_dir, city):
    city_path = os.path.join(root_dir, city)

    past_dir = os.path.join(city_path, "2015")
    future_dir = os.path.join(city_path, "2020")

    past_files = sorted(os.listdir(past_dir))
    future_files = sorted(os.listdir(future_dir))

    past_images = []
    future_images = []

    # pair by index, NOT filename
    n = min(len(past_files), len(future_files))

    for i in range(n):
        past_path = os.path.join(past_dir, past_files[i])
        future_path = os.path.join(future_dir, future_files[i])

        past_img = transform(Image.open(past_path).convert("RGB"))
        future_img = transform(Image.open(future_path).convert("RGB"))

        past_images.append(past_img)
        future_images.append(future_img)

    return past_images, future_images

load_city_images(root_dir, "Chicago")

([tensor([[[0., 0., 0.,  ..., 0., 0., 0.],
           [0., 0., 0.,  ..., 0., 0., 0.],
           [0., 0., 0.,  ..., 0., 0., 0.],
           ...,
           [0., 0., 0.,  ..., 0., 0., 0.],
           [0., 0., 0.,  ..., 0., 0., 0.],
           [0., 0., 0.,  ..., 0., 0., 0.]],
  
          [[0., 0., 0.,  ..., 0., 0., 0.],
           [0., 0., 0.,  ..., 0., 0., 0.],
           [0., 0., 0.,  ..., 0., 0., 0.],
           ...,
           [0., 0., 0.,  ..., 0., 0., 0.],
           [0., 0., 0.,  ..., 0., 0., 0.],
           [0., 0., 0.,  ..., 0., 0., 0.]],
  
          [[0., 0., 0.,  ..., 0., 0., 0.],
           [0., 0., 0.,  ..., 0., 0., 0.],
           [0., 0., 0.,  ..., 0., 0., 0.],
           ...,
           [0., 0., 0.,  ..., 0., 0., 0.],
           [0., 0., 0.,  ..., 0., 0., 0.],
           [0., 0., 0.,  ..., 0., 0., 0.]]])],
 [tensor([[[0., 0., 0.,  ..., 0., 0., 0.],
           [0., 0., 0.,  ..., 0., 0., 0.],
           [0., 0., 0.,  ..., 0., 0., 0.],
           ...,
           [0., 0., 0

In [ ]:
root_dir = "C:/Users/adamk/Downloads/SatelliteImages/"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

city = input("Enter a city name (e.g., Atlanta, Chicago, Houston): ").strip().lower()

import os
from PIL import Image
from torchvision import transforms
import torch

transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor()
])

def load_city_images(root_dir, city):
    city_path = os.path.join(root_dir, city)

    past_dir = os.path.join(city_path, "2015")
    future_dir = os.path.join(city_path, "2020")

    past_files = sorted(os.listdir(past_dir))
    future_files = sorted(os.listdir(future_dir))

    past_images = []
    future_images = []

    # pair by index, NOT filename
    n = min(len(past_files), len(future_files))

    for i in range(n):
        past_path = os.path.join(past_dir, past_files[i])
        future_path = os.path.join(future_dir, future_files[i])

        past_img = transform(Image.open(past_path).convert("RGB"))
        future_img = transform(Image.open(future_path).convert("RGB"))

        past_images.append(past_img)
        future_images.append(future_img)

    return past_images, future_images

import pandas as pd
from skimage.metrics import structural_similarity as ssim

def evaluate_city(fake_img, real_imgs):
    real_img = real_imgs[0]  # simple baseline comparison

    fake_np = fake_img.permute(1,2,0).numpy()
    real_np = real_img.permute(1,2,0).numpy()

    ssim_score = ssim(fake_np, real_np, channel_axis=2, data_range=1.0)

    results = pd.DataFrame({
        "Metric": ["SSIM"],
        "Value": [ssim_score]
    })

    return results

import os
from torchvision.utils import save_image

def save_to_downloads(image_tensor, city):
    image_tensor = image_tensor.clamp(0,1)

    downloads_path = os.path.join(os.path.expanduser("~"), "C:/Users/adamk/Downloads/SatelliteImages/")
    filename = f"{city}_generated.png"
    full_path = os.path.join(downloads_path, filename)

    save_image(image_tensor, full_path)

    print(f"Saved image to: {full_path}")

import matplotlib.pyplot as plt

def show_image(img_tensor):
    img = img_tensor.permute(1,2,0).numpy()
    plt.imshow(img)
    plt.axis("off")
    plt.show()

def generate_city_image(G, past_images):
    G.eval()

    with torch.no_grad():
        img = past_images[0].unsqueeze(0).to(device)

        out = G(img)

        out = torch.clamp(img + out, 0, 1)

    return out.squeeze(0).cpu()

#______________________________________________________________________________


G = Generator().to(device)
G.eval()

def run_pipeline(G, root_dir):

    # 2. Load data
    past_images, real_images = load_city_images(root_dir, city)

    if len(past_images) == 0:
        print("No data found for this city.")
        return

    # 3. Generate image
    fake_image = generate_city_image(G, past_images)

    # 4. Evaluate
    results_table = evaluate_city(fake_image, real_images)

    # 5. Print results
    print("\n=== Evaluation Metrics ===")
    print(results_table)

    # 6. Save image
    save_to_downloads(fake_image, city)

    print("\nPipeline complete ✔")


run_pipeline(G, root_dir)

Using device: cpu

=== Evaluation Metrics ===
  Metric     Value
0   SSIM  0.607764
Saved image to: C:/Users/adamk/Downloads/SatelliteImages/washington_generated.png

Pipeline complete ✔
